In [1]:
# Input:  silver_master_sales
# Output: silver_features
from pyspark.sql import functions as F
from pyspark.sql.window import Window


StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 3, Finished, Available, Finished, False)

In [2]:
df = spark.read.format("delta").table("silver_master_sales")
print(f"Input rows: {df.count():,}")

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 4, Finished, Available, Finished, False)

Input rows: 3,000,888


## Per-store, per-family time window

In [3]:
# This window partitions data so lag features don't bleed
# across different stores or product families
w = Window.partitionBy("store_nbr", "family").orderBy("date")

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 5, Finished, Available, Finished, False)

## Lag Features

In [4]:
# Lag = "what was the sales value N days ago for this same
# store + family?" — the single most predictive feature set
# for time-series retail forecasting.
df_lag = (
    df
    .withColumn("lag_1",  F.lag("sales", 1).over(w))   # yesterday
    .withColumn("lag_7",  F.lag("sales", 7).over(w))   # same weekday last week
    .withColumn("lag_14", F.lag("sales", 14).over(w))  # 2 weeks ago
    .withColumn("lag_28", F.lag("sales", 28).over(w))  # 4 weeks ago (same weekday)
    .withColumn("lag_364",F.lag("sales", 364).over(w)) # last year same day
    # Promotional lags
    .withColumn("promo_lag_1",  F.lag("onpromotion", 1).over(w))
    .withColumn("promo_lag_7",  F.lag("onpromotion", 7).over(w))
)

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 6, Finished, Available, Finished, False)

##  Rolling Statistics

In [5]:
# Rolling averages smooth day-to-day noise and give the model
# a sense of recent trend direction.
df_roll = (
    df_lag
    .withColumn("roll_avg_7",
        F.avg("sales").over(w.rowsBetween(-7, -1)))
    .withColumn("roll_avg_14",
        F.avg("sales").over(w.rowsBetween(-14, -1)))
    .withColumn("roll_avg_28",
        F.avg("sales").over(w.rowsBetween(-28, -1)))
    .withColumn("roll_std_7",
        F.stddev("sales").over(w.rowsBetween(-7, -1)))
    .withColumn("roll_std_14",
        F.stddev("sales").over(w.rowsBetween(-14, -1)))
    .withColumn("roll_max_7",
        F.max("sales").over(w.rowsBetween(-7, -1)))
    .withColumn("roll_min_7",
        F.min("sales").over(w.rowsBetween(-7, -1)))
    .withColumn("roll_median_7",
        F.percentile_approx("sales", 0.5).over(w.rowsBetween(-7, -1)))
    # Trend: is this week higher or lower than last week?
    .withColumn("trend_7v14",
        F.col("roll_avg_7") - F.col("roll_avg_14"))
)

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 7, Finished, Available, Finished, False)

## Store-level Context Features

In [6]:
# How busy is the WHOLE STORE on a given day?
# (A busy store day lifts all categories)
store_day_w = Window.partitionBy("store_nbr", "date")
 
df_ctx = (
    df_roll
    .withColumn("store_day_total",
        F.sum("sales").over(store_day_w))
    .withColumn("store_day_txn_total",
        F.sum("transactions").over(store_day_w))
    # Family's share of store total
    .withColumn("family_share_pct",
        F.col("sales") / (F.col("store_day_total") + F.lit(1e-6)))
    # Store cluster average (how do peer stores perform?)
    .withColumn("cluster_avg_sales",
        F.avg("sales").over(Window.partitionBy("cluster","date","family")))
)

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 8, Finished, Available, Finished, False)

## Encode Categorical Columns

In [7]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
 
indexers = [
    StringIndexer(inputCol="family",     outputCol="family_idx",     handleInvalid="keep"),
    StringIndexer(inputCol="store_type", outputCol="store_type_idx", handleInvalid="keep"),
    StringIndexer(inputCol="city",       outputCol="city_idx",       handleInvalid="keep"),
]
pipeline = Pipeline(stages=indexers)
model    = pipeline.fit(df_ctx)
df_enc   = model.transform(df_ctx)

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 9, Finished, Available, Finished, False)

## Drop startup nulls from lag window

In [8]:
# Rows in the first 28 days of each store+family series won't
# have valid lag_28 — drop them to avoid feeding NaN to ML.
REQUIRED_LAGS = ["lag_7", "lag_14", "lag_28", "roll_avg_7"]
 
df_features = df_enc.dropna(subset=REQUIRED_LAGS)
 
# Fill remaining nulls (lag_1, lag_364 for early rows)
df_features = df_features.fillna({
    "lag_1": 0, "lag_364": 0,
    "promo_lag_1": 0, "promo_lag_7": 0,
    "roll_std_7": 0, "roll_std_14": 0,
    "cluster_avg_sales": 0,
})
 
print(f"After dropping null-lag startup rows: {df_features.count():,}")

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 10, Finished, Available, Finished, False)

After dropping null-lag startup rows: 2,950,992


## Final Feature List (for documentation)

In [9]:
FEATURE_COLS = [
    # Lag features
    "lag_1", "lag_7", "lag_14", "lag_28", "lag_364",
    # Rolling stats
    "roll_avg_7", "roll_avg_14", "roll_avg_28",
    "roll_std_7", "roll_std_14",
    "roll_max_7", "roll_min_7", "roll_median_7", "trend_7v14",
    # Promotions
    "onpromotion", "promo_lag_1", "promo_lag_7",
    # Context
    "oil_price", "is_holiday",
    "store_day_total", "store_day_txn_total",
    "family_share_pct", "cluster_avg_sales",
    # Date parts
    "year", "month", "day", "weekday", "week_of_year",
    "quarter", "is_month_end", "is_month_start",
    # Encoded categories
    "family_idx", "store_type_idx", "city_idx", "cluster",
]
 
print(f"\nTotal features: {len(FEATURE_COLS)}")
for f in FEATURE_COLS:
    print(f"  + {f}")

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 11, Finished, Available, Finished, False)


Total features: 35
  + lag_1
  + lag_7
  + lag_14
  + lag_28
  + lag_364
  + roll_avg_7
  + roll_avg_14
  + roll_avg_28
  + roll_std_7
  + roll_std_14
  + roll_max_7
  + roll_min_7
  + roll_median_7
  + trend_7v14
  + onpromotion
  + promo_lag_1
  + promo_lag_7
  + oil_price
  + is_holiday
  + store_day_total
  + store_day_txn_total
  + family_share_pct
  + cluster_avg_sales
  + year
  + month
  + day
  + weekday
  + week_of_year
  + quarter
  + is_month_end
  + is_month_start
  + family_idx
  + store_type_idx
  + city_idx
  + cluster


## Write Silver Features Table

In [10]:
df_features.write.format("delta").mode("overwrite") \
           .option("overwriteSchema", "true") \
           .saveAsTable("silver_features")
 
print(f"\n✅ silver_features written: {df_features.count():,} rows")
print("Proceed to 04_ml_experiment.py")

StatementMeta(, 2896dae8-0bbc-4005-a5c0-ba545efc3bff, 12, Finished, Available, Finished, False)


✅ silver_features written: 2,950,992 rows
Proceed to 04_ml_experiment.py
